# <font color="#418FDE" size="6.5" uppercase>**Bagging Embeddings**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Apply bagging classifiers and regressors with different base estimators. 
- Compare bootstrap, pasting, random subspace, and random patch strategies. 
- Use random tree embeddings as feature transformations for downstream models. 


## **1. Bagging Models**

### **1.1. Bagging Classifier**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_17/Lecture_B/image_01_01.jpg?v=1784010323" width="250">



>* Train classifiers on resampled training data
>* Combine votes for more stable predictions

>* Bootstrap diversity creates useful model disagreement
>* Voting improves robustness and generalization

>* Bagging supports many base classifiers
>* Balance flexibility, variance reduction, and cost



In [ ]:
#@title Python Code - Bagging Classifier

# This example trains a bagging classifier.
# Bootstrap samples create diverse decision trees.
# Voting usually improves test accuracy stability.

import sklearn
from sklearn.datasets import make_classification
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Create a small synthetic classification dataset.
features, labels = make_classification(
    n_samples=800,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    random_state=42,
)

# Check that features and labels match correctly.
if features.shape[0] != labels.shape[0]:
    raise ValueError("Feature rows must match label count.")

# Split data while preserving class proportions.
train_features, test_features, train_labels, test_labels = train_test_split(
    features,
    labels,
    test_size=0.30,
    stratify=labels,
    random_state=42,
)

# Train one flexible decision tree for comparison.
single_tree = DecisionTreeClassifier(random_state=42)
single_tree.fit(train_features, train_labels)

# Train many bootstrapped trees inside one bagging classifier.
bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=50,
    max_samples=0.80,
    bootstrap=True,
    random_state=42,
)

bagging_model.fit(train_features, train_labels)

# Compare predictions from one tree and the bagged ensemble.
tree_predictions = single_tree.predict(test_features)
bagging_predictions = bagging_model.predict(test_features)

# Accuracy shows how often each model predicts correctly.
tree_accuracy = accuracy_score(test_labels, tree_predictions)
bagging_accuracy = accuracy_score(test_labels, bagging_predictions)

print("scikit-learn version:", sklearn.__version__)
print("Single decision tree accuracy:", round(tree_accuracy, 3))
print("Bagging classifier accuracy:", round(bagging_accuracy, 3))



### **1.2. Bagging Regressor**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_17/Lecture_B/image_01_02.jpg?v=1784010325" width="250">



>* Averages resampled regressors for continuous predictions
>* Reduces variance in flexible models like trees

>* Train regressors on resampled data subsets
>* Average predictions for smoother estimates

>* Choose diverse, flexible base regressors
>* Bagging reduces variance, not missing patterns



In [ ]:
#@title Python Code - Bagging Regressor

# Demonstrate bagging for continuous prediction.
# Compare one tree with averaged trees.
# Expect lower test error from bagging.

import numpy as np
import sklearn
from sklearn.datasets import make_regression

from sklearn.ensemble import BaggingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

# Create a small noisy regression dataset.
features, target = make_regression(
    n_samples=600,
    n_features=6,
    noise=25.0,
    random_state=42,
)

# Confirm the feature matrix matches the target length.
if features.shape[0] != target.shape[0]:
    raise ValueError("Features and target must have matching rows.")

# Split data before fitting any model.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.25,
    random_state=42,
)

# Fit one flexible tree as the baseline.
single_tree = DecisionTreeRegressor(random_state=42)
single_tree.fit(X_train, y_train)

# Fit many trees on bootstrap samples and average them.
bagging_model = BaggingRegressor(
    estimator=DecisionTreeRegressor(random_state=42),
    n_estimators=50,
    random_state=42,
)

bagging_model.fit(X_train, y_train)

# Evaluate both models on unseen test data.
tree_predictions = single_tree.predict(X_test)
bagging_predictions = bagging_model.predict(X_test)

# Mean absolute error is in target units.
tree_mae = mean_absolute_error(y_test, tree_predictions)
bagging_mae = mean_absolute_error(y_test, bagging_predictions)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Single decision tree MAE: {tree_mae:.1f}")
print(f"Bagging regressor MAE: {bagging_mae:.1f}")
print(f"Bagging used {bagging_model.n_estimators} bootstrap-trained trees.")



### **1.3. Base Estimator**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_17/Lecture_B/image_01_03.jpg?v=1784010327" width="250">



>* Base estimators train on resampled data
>* Combined predictions reduce unstable model errors

>* Bagging works with many estimator types
>* Diverse models smooth unstable predictions

>* Favor high-variance estimators for bagging
>* Match estimator choice to task needs



In [ ]:
#@title Python Code - Base Estimator

# Compare base estimators inside one bagging classifier.
# Bagging copies the chosen learner many times.
# Test accuracy shows how the choice matters.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Check that features and labels match.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match label count.")

# Split once so both ensembles face the same test cases.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Build two bagging models with different base estimators.
tree_bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=30,
    random_state=42,
)

knn_bag = BaggingClassifier(
    estimator=KNeighborsClassifier(n_neighbors=5),
    n_estimators=30,
    random_state=42,
)

# Fit each ensemble on resampled versions of the training data.
tree_bag.fit(X_train, y_train)
knn_bag.fit(X_train, y_train)

# Evaluate how the base estimator changes ensemble behavior.
tree_accuracy = accuracy_score(y_test, tree_bag.predict(X_test))
knn_accuracy = accuracy_score(y_test, knn_bag.predict(X_test))

print("scikit-learn version:", sklearn.__version__)
print("Bagging base estimator comparison on breast cancer data")
print("Decision tree base accuracy:", round(tree_accuracy, 3))
print("K-nearest neighbors base accuracy:", round(knn_accuracy, 3))
print("Same bagging idea, different copied learner, different result.")



## **2. Sampling Strategies**

### **2.1. Bootstrap Versus Pasting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_17/Lecture_B/image_02_01.jpg?v=1784010319" width="250">



>* Bootstrap samples with replacement, allowing repeats
>* Pasting samples without replacement for diversity

>* Bootstrap repeats records to diversify tree models
>* Pasting samples without duplicates, avoiding extra weight

>* Bootstrap adds diversity and supports out-of-bag checks
>* Pasting avoids duplicates for sensitive datasets



In [ ]:
#@title Python Code - Bootstrap Versus Pasting

# Compare bootstrap and pasting samples.
# Replacement controls duplicates within each sample.
# The printed counts reveal the difference.

import numpy as np

# Use a tiny dataset of labeled training examples.
training_examples = np.array(["A", "B", "C", "D", "E", "F"])
sample_size = 6

# Make the random draws reproducible for the lecture.
rng = np.random.default_rng(42)

# Bootstrap samples with replacement, so repeats are possible.
bootstrap_sample = rng.choice(
    training_examples,
    size=sample_size,
    replace=True,
)

# Pasting samples without replacement, so repeats are impossible.
pasting_sample = rng.choice(
    training_examples,
    size=sample_size,
    replace=False,
)

# Count unique examples and omitted examples in each sample.
bootstrap_unique = len(set(bootstrap_sample))
pasting_unique = len(set(pasting_sample))

bootstrap_omitted = sorted(set(training_examples) - set(bootstrap_sample))
pasting_omitted = sorted(set(training_examples) - set(pasting_sample))

print("Original examples: " + ", ".join(training_examples))
print("Bootstrap sample: " + ", ".join(bootstrap_sample))
print("Pasting sample: " + ", ".join(pasting_sample))
print("Bootstrap unique examples: " + str(bootstrap_unique) + " of 6")
print("Pasting unique examples: " + str(pasting_unique) + " of 6")
print("Bootstrap omitted examples: " + str(bootstrap_omitted))
print("Pasting omitted examples: " + str(pasting_omitted))
print("Key idea: bootstrap allows repeats; pasting does not.")



### **2.2. Random Feature Subsets**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_17/Lecture_B/image_02_02.jpg?v=1784010317" width="250">



>* Train models on different feature subsets
>* Combine diverse views for stable predictions

>* Useful for many noisy or redundant features
>* Reveals diverse patterns beyond dominant predictors

>* Subset size balances bias, variance, and speed
>* Diverse learners improve robust ensemble generalization



In [ ]:
#@title Python Code - Random Feature Subsets

# Compare row and feature sampling strategies.
# Random subspaces train learners on feature subsets.
# Accuracy shows how sampling choices affect ensembles.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Create a small dataset with useful and noisy features.
features, target = make_classification(
    n_samples=900, n_features=12, n_informative=5, random_state=42
)

# Split data so every strategy uses the same test set.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.3, stratify=target, random_state=42
)

# Validate the feature count before sampling columns.
if X_train.shape[1] < 4:
    raise ValueError("This lesson needs at least four features.")

base_tree = DecisionTreeClassifier(max_depth=4, random_state=42)

strategies = {
    "Bootstrap rows": (True, 1.0),
    "Pasting rows": (False, 1.0),
    "Random subspace": (False, 0.5),
    "Random patches": (True, 0.5),
}

print("scikit-learn version:", sklearn.__version__)
print("Strategy | row bootstrap | feature fraction | accuracy")

for strategy_name, settings in strategies.items():
    use_bootstrap, feature_fraction = settings
    model = BaggingClassifier(
        estimator=base_tree,
        n_estimators=40,
        max_samples=0.7,
        max_features=feature_fraction,
        bootstrap=use_bootstrap,
        bootstrap_features=False,
        random_state=42,
    )
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    print(
        strategy_name,
        "|",
        use_bootstrap,
        "|",
        feature_fraction,
        "|",
        round(accuracy, 3),
    )

print("Random subspace changes columns; bootstrap and pasting change rows.")



### **2.3. Random Patch Sampling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_17/Lecture_B/image_02_03.jpg?v=1784010321" width="250">



>* Sample both rows and features
>* Combine varied views for stable predictions

>* Boosts model diversity and lowers training cost
>* Combines partial views despite noisy features

>* Patch size balances model strength and diversity
>* Best for wide data, risky for small datasets



In [ ]:
#@title Python Code - Random Patch Sampling

# This example demonstrates random patch sampling.
# Each learner receives different rows and features.
# The printed patches reveal ensemble diversity.

import numpy as np
import sklearn
from sklearn.datasets import make_classification

# Create a small dataset with named feature columns.
features, target = make_classification(
    n_samples=120,
    n_features=8,
    n_informative=4,
    n_redundant=1,
    random_state=42,
)

# Validate the dataset shape before sampling patches.
if features.shape != (120, 8):
    raise ValueError("Expected 120 rows and 8 features.")

rng = np.random.default_rng(42)
feature_names = np.array(["f0", "f1", "f2", "f3", "f4", "f5", "f6", "f7"])

# Build three random patches using rows and columns.
patch_count = 3
row_fraction = 0.55
feature_fraction = 0.50

row_count = int(features.shape[0] * row_fraction)
feature_count = int(features.shape[1] * feature_fraction)

print("scikit-learn version:", sklearn.__version__)
print("Full data shape: 120 rows x 8 features")
print("Each patch uses 66 rows x 4 features")

for patch_number in range(1, patch_count + 1):
    row_indices = rng.choice(features.shape[0], row_count, replace=True)
    column_indices = rng.choice(features.shape[1], feature_count, replace=False)
    selected_names = feature_names[np.sort(column_indices)]
    unique_rows = len(np.unique(row_indices))
    names_text = ", ".join(selected_names.tolist())
    print(f"Patch {patch_number}: {unique_rows} unique rows, features {names_text}")

print("Random patches vary both observations and feature columns.")



## **3. Embedding Diversity**

### **3.1. Out of Bag Prediction**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_17/Lecture_B/image_03_01.jpg?v=1784010311" width="250">



>* Estimate performance without separate validation data
>* Use left-out samples for honest predictions

>* OOB checks tree feature quality
>* Aggregated assignments reveal stable embedding patterns

>* OOB checks whether tree diversity helps
>* Strong OOB results suggest transferable embeddings



In [ ]:
#@title Python Code - Out of Bag Prediction

# Demonstrate out of bag prediction with embeddings.
# Compare honest OOB features with training features.
# Show how embedded features support classification.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.ensemble import BaggingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Create a small deterministic classification dataset.
features, target = make_classification(
    n_samples=800,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    random_state=42,
)

# Split once so the final test set stays unseen.
train_features, test_features, train_target, test_target = train_test_split(
    features,
    target,
    test_size=0.30,
    stratify=target,
    random_state=42,
)

# Bagged trees provide bootstrap samples and OOB predictions.
base_tree = DecisionTreeClassifier(max_depth=4, random_state=42)
bagged_trees = BaggingClassifier(
    estimator=base_tree,
    n_estimators=80,
    bootstrap=True,
    oob_score=True,
    random_state=42,
)

# Fit the ensemble before extracting tree leaf embeddings.
bagged_trees.fit(train_features, train_target)

# Validate that every training row received an OOB estimate.
oob_probabilities = bagged_trees.oob_decision_function_
valid_oob_rows = np.isfinite(oob_probabilities).all(axis=1)
if valid_oob_rows.sum() != len(train_target):
    raise ValueError("Increase n_estimators so every row has OOB predictions.")

# Convert each example into its leaf index per tree.
train_leaf_features = np.column_stack(
    [
        tree.apply(train_features[:, feature_indices])
        for tree, feature_indices in zip(
            bagged_trees.estimators_, bagged_trees.estimators_features_
        )
    ]
)
test_leaf_features = np.column_stack(
    [
        tree.apply(test_features[:, feature_indices])
        for tree, feature_indices in zip(
            bagged_trees.estimators_, bagged_trees.estimators_features_
        )
    ]
)

# One-hot encode leaf indices into random tree embedding features.
train_embedding = np.hstack(
    [
        tree.decision_path(train_features[:, feature_indices]).toarray()
        for tree, feature_indices in zip(
            bagged_trees.estimators_, bagged_trees.estimators_features_
        )
    ]
)
test_embedding = np.hstack(
    [
        tree.decision_path(test_features[:, feature_indices]).toarray()
        for tree, feature_indices in zip(
            bagged_trees.estimators_, bagged_trees.estimators_features_
        )
    ]
)

# Train a simple downstream model on the tree embedding.
downstream_model = LogisticRegression(max_iter=1000, random_state=42)
downstream_model.fit(train_embedding, train_target)

# Compare OOB ensemble accuracy with downstream test accuracy.
oob_predictions = np.argmax(oob_probabilities, axis=1)
test_predictions = downstream_model.predict(test_embedding)
oob_accuracy = accuracy_score(train_target, oob_predictions)
test_accuracy = accuracy_score(test_target, test_predictions)

# Print concise results for the teaching point.
print("scikit-learn version:", sklearn.__version__)
print("OOB accuracy from bagged trees:", round(oob_accuracy, 3))
print("Test accuracy after tree embedding:", round(test_accuracy, 3))
print("Embedding columns created:", train_embedding.shape[1])



### **3.2. Random Tree Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_17/Lecture_B/image_03_02.jpg?v=1784010315" width="250">



>* Trees convert data into leaf-based features
>* Embeddings capture nonlinear interactions automatically

>* Randomness creates diverse tree-based features
>* Downstream models combine richer region signals

>* Feed tree embeddings into downstream models
>* Reveal nonlinear patterns through learned regions



In [ ]:
#@title Python Code - Random Tree Features

# Random trees can create useful feature embeddings.
# Leaf indicators encode nonlinear regions of data.
# Logistic regression improves after tree transformation.

import numpy as np
import sklearn
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomTreesEmbedding
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Make a small nonlinear classification dataset.
features, target = make_moons(n_samples=600, noise=0.25, random_state=42)

# Check that the generated data has the expected shape.
if features.shape != (600, 2):
    raise ValueError("Expected 600 rows and 2 input features.")

# Split before fitting any transformation or model.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.3, stratify=target, random_state=42
)

# A plain linear model sees only the original coordinates.
plain_model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
)
plain_model.fit(X_train, y_train)
plain_accuracy = accuracy_score(y_test, plain_model.predict(X_test))

# Random tree embedding turns each reached leaf into a feature.
tree_embedding = RandomTreesEmbedding(
    n_estimators=40, max_depth=3, random_state=42
)
embedded_model = make_pipeline(
    tree_embedding, LogisticRegression(max_iter=1000, random_state=42)
)
embedded_model.fit(X_train, y_train)
embedded_accuracy = accuracy_score(y_test, embedded_model.predict(X_test))

# Inspect the transformed feature space created by the trees.
embedded_train = tree_embedding.transform(X_train)
active_leaf_features = int(embedded_train[0].sum())

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Original feature count: {X_train.shape[1]}")
print(f"Random tree feature count: {embedded_train.shape[1]}")
print(f"Active leaf indicators per sample: {active_leaf_features}")
print(f"Plain logistic accuracy: {plain_accuracy:.3f}")
print(f"Tree-embedded logistic accuracy: {embedded_accuracy:.3f}")



### **3.3. Bias Variance Diversity**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_17/Lecture_B/image_03_03.jpg?v=1784010313" width="250">



>* Random trees balance bias, variance, and diversity
>* Leaf memberships become features for downstream models

>* Diverse trees reveal different data patterns
>* Embeddings help simple models use complex interactions

>* Tree depth controls specificity and noise
>* Regularized models select reliable embedded patterns



# <font color="#418FDE" size="6.5" uppercase>**Bagging Embeddings**</font>


In this lecture, you learned to:
- Apply bagging classifiers and regressors with different base estimators. 
- Compare bootstrap, pasting, random subspace, and random patch strategies. 
- Use random tree embeddings as feature transformations for downstream models. 

In the next Module (Module 18), we will go over 'Boosting Stacking'